# FPGA-Accelerated Lane Detection with Confidence-Controlled Edge Thresholding
### Major Project — 7th Semester B.E. Electronics & Communication Engineering (ECE)
**Institution**: Panjab University, Chandigarh (UIET)  
**Team Members**:  
- Tarun Kumar Meharda (Roll No: UE235108)  
- Sarthak Verma (Roll No: UE235098)  
**Project Guide**: Dr. Arvind Kumar  
**Hardware Target**: PYNQ-Z2 Development Board (AMD Xilinx Zynq-7000 SoC `XC7Z020`)

## 1. System Architecture: PS-PL Hardware/Software Co-Design
The system partitions driver-assistance vision tasks across the **Processing System (PS)** and **Programmable Logic (PL)**:
- **PL (FPGA Fabric)**: Executes computationally heavy pixel-rate preprocessing at 100 MHz (RGB-to-Gray fixed point, 2-line buffering, 3x3 Sobel convolution, and dynamic threshold comparator).
- **PS (ARM Cortex-A9)**: Manages AXI DMA contiguous buffers, high-level polynomial curve fitting ($x = ay^2 + by + c$), marking-quality scoring ($Q = w_C C + w_E E + w_G G$), virtual-lane fallback, and speed-breaker warnings.
- **Feedback Control Loop**: The ARM processor calculates $Q$ and writes the updated threshold back to the FPGA over the **AXI4-Lite register bus (`0x43C00010`)** for real-time adaptation to shadows and faded paint.

In [ ]:
import os
import sys
import time
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Ensure project directory is in sys.path
CURRENT_DIR = os.getcwd()
if CURRENT_DIR not in sys.path:
    sys.path.insert(0, CURRENT_DIR)

from lane_hardware_driver import PYNQLaneDetectorDriver
from cpu_baseline.lane_detection import LaneDetector, LaneStatus
from cpu_baseline.confidence_score import ConfidenceScorer
from cpu_baseline.virtual_lane import VirtualLaneEstimator, PipelineMode
from cpu_baseline.speed_breaker import SpeedBreakerDetector

print("Libraries successfully imported!")

## 2. Initialize PYNQ Hardware Driver
Instantiate the driver. When running on the physical PYNQ-Z2 board, this loads `lane_accel.bit` into the FPGA fabric and sets up AXI DMA buffers. On a host PC, it seamlessly runs the bit-accurate hardware emulation model.

In [ ]:
# Initialize Driver (640x360 processing resolution)
driver = PYNQLaneDetectorDriver(frame_width=640, frame_height=360)

# Read hardware geometry from AXI-Lite register 0x0C
width, height = driver.get_geometry()
print(f"Hardware Resolution: {width} x {height}")
print(f"Initial Edge Threshold (Reg 0x10): T = {driver.get_threshold()}")

## 3. Dynamic Threshold Control via AXI-Lite
Demonstrate updating the threshold register over the AXI-Lite bus in real-time.

In [ ]:
# Write a new threshold T = 75
driver.set_threshold(75)
updated_t = driver.get_threshold()
print(f"Successfully wrote and verified AXI-Lite threshold: T = {updated_t}")

# Reset to standard default (50)
driver.set_threshold(50)

## 4. Single-Frame Acceleration Demo: Input to FPGA Binary Edges
Process a synthetic road frame through the hardware accelerator and display the resulting edge detection.

In [ ]:
from cpu_baseline.test_synthetic_pipeline import generate_synthetic_road_image

# Generate realistic input frame
test_frame = generate_synthetic_road_image(640, 360)

# Execute on FPGA Hardware
t_start = time.perf_counter()
hw_edge_map = driver.process_frame_hardware(test_frame)
t_elapsed_ms = (time.perf_counter() - t_start) * 1000.0

print(f"FPGA Execution Latency: {t_elapsed_ms:.2f} ms ({1000.0/t_elapsed_ms:.1f} FPS)")

# Visual Display
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(test_frame, cv2.COLOR_BGR2RGB))
axes[0].set_title("Input Camera Video Frame (24-bit RGB)")
axes[0].axis("off")

axes[1].imshow(hw_edge_map, cmap="gray")
axes[1].set_title("FPGA Hardware Output (8-bit Binary Edge Map)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 5. End-to-End Pipeline: Lane Fitting, Scoring & HUD Overlay
Pass the FPGA edge map into the ARM software modules to fit curves, score confidence, and render the lane corridor.

In [ ]:
from cpu_baseline.preprocessing import apply_roi_mask

detector = LaneDetector(640, 360)
scorer = ConfidenceScorer()

# Apply Region of Interest mask to hardware edges
roi_edges = apply_roi_mask(hw_edge_map)

# PS: Fit polynomial curves
geom, debug_info = detector.process_frame(roi_edges)

# PS: Calculate marking quality score
quality = scorer.evaluate_quality(roi_edges, hw_edge_map, geom, debug_info)

# Render complete annotated HUD overlay
overlay_img = detector.render_overlay(test_frame, geom, debug_info)

print(f"Lane Status:        {geom.status.value}")
print(f"Marking Quality Q:  {quality.q_score:.3f} [{quality.band.value}]")
print(f"Lateral Offset:     {geom.lateral_offset_px:+.1f} px")
print(f"Detected Width:     {geom.lane_width_bottom_px:.1f} px")

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(overlay_img, cv2.COLOR_BGR2RGB))
plt.title(f"Complete Autonomous Navigation HUD Overlay | Q = {quality.q_score:.2f}")
plt.axis("off")
plt.show()

## 6. Performance Benchmark: CPU vs. FPGA Accelerator
Execute benchmark iterations comparing the CPU preprocessing baseline against FPGA hardware execution.

In [ ]:
bench = driver.benchmark_comparison(test_frame, iterations=5)

# Plot Bar Chart Comparison
labels = ['CPU Baseline Preprocessing', 'FPGA Hardware Acceleration']
latencies = [bench['cpu_mean_ms'], bench['fpga_mean_ms']]
colors = ['#d9534f', '#5cb85c']

plt.figure(figsize=(8, 4.5))
bars = plt.bar(labels, latencies, color=colors, width=0.5)
plt.ylabel('Processing Latency (ms / frame)')
plt.title(f"Latency Benchmark Comparison (Speedup: {bench['speedup']:.2f}x)")
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.3, f"{yval:.2f} ms", ha='center', va='bottom', fontweight='bold')
plt.ylim(0, max(latencies) * 1.3)
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.show()

## 7. Clean Up Contiguous Memory
Release physical DMA buffer memory allocations before exiting.

In [ ]:
driver.close()
print("Hardware driver closed and contiguous buffers released safely.")